# ARCHIVED RESEARCH NOTE

**What this notebook works on:** Tests spatial operator consistency by training a coupled grid operator at 16 positions then evaluating unchanged parameters at 32/64/128, comparing scaled/unscaled, spectral CCNN, FNO-style and attention controls under local/nonlocal generators.

**Research question:** RQ4/H4: Does grid-aware coupled evolution transfer accurately across resolutions and horizons, and is any benefit unique relative to strong resolution-neutral baselines?

**Output:** On execution: rq4_raw_metrics.csv, rq4_training_metrics.csv, rq4_summary.csv, transfer plots, rq4_hypothesis_ledger.csv/.json, rq4_paired_uncertainty.csv, RQ4_CONCLUSION_AND_NEXT_RESEARCH.md and rq4_run_manifest.json. The separate rq4_results_preliminary folder contains 3-seed synthetic CPU findings: scaled > unscaled; stronger spectral baseline prevents a broad CCNN-win claim.

**Evidence status:** Expected outputs are not newly measured by this archival operation. Separate preliminary results, when named above, remain exploratory; no unique CCNN superiority is implied. Original cells were preserved.

**Original Git blob:** `7f0c5be3b6160d08c7cd5c58124282de1aa7ab07`

See the companion note and archive index in `notebooks/archive/`.

# RQ4 — Spatial-Resolution Transfer of Coupled Continuous Neural Fields
### An operator-consistency test for attention augmentation and future weight-space repair

**Research identity:** [CCNN × attention × verified weight-space repair charter](https://github.com/mansura-habiba/ode_nn/blob/master/notebooks/RESEARCH_QUESTIONS_CCNN_ATTENTION_WEIGHT_REPAIR.md) · **RQ4 / H4** · experimental protocol v1 · *Results must be generated by running this notebook.*

## Abstract

Whether coupled continuous convolutional neural networks (CCNNs) provide a useful inductive bias beyond generic Neural ODEs and attention models remains unresolved, particularly when the spatial resolution of a system changes after training. This study isolates one testable architectural property: **transfer of shared learned dynamical parameters across spatial discretizations without retraining**. We construct smooth periodic initial fields from continuous Fourier coefficients and generate reference trajectories at multiple grid resolutions from the same underlying linear field–global-state system. All methods are trained exclusively on 16 spatial channels and evaluated at 16, 32, 64, and 128 channels. The proposed operator is a grid-aware conservative discretization of a coupled diffusion–reaction PDE and global ODE, contrasted against an unscaled convolutional operator, a spectrally consistent coupled operator, a resolution-neutral pointwise Neural ODE, a compact Fourier neural operator, and coordinate-aware attention. To detect generator-specific advantage, we repeat the experiment on nonlocal interactions whose Fourier symbol is incompatible with a simple diffusion operator. We compare pointwise error, relative spatial error, temporal extrapolation, stability, and wall-clock evaluation cost. A machine-generated evidence ledger classifies hypotheses as supported, contradicted, inconclusive, or untested **within the scope of the executed toy experiment**. The work is intended as a falsification-oriented precursor to evaluating CCNN-augmented attention and cross-resolution weight-space repair; it does not assume improved performance.

## 1. Motivation and problem statement

Earlier irregular-time experiments found that continuous integration helped long-gap prediction but **did not establish that coupled CCNN outperforms a generic Neural ODE**. A second observation showed that the effect did not generalize to a nonlocal system. We therefore test whether a different potential advantage—the consistency of a **shared spatial operator across grids**—can be demonstrated. This is a more stringent test than fitting the dynamics at a fixed resolution.

**RQ4:** After learning dynamics at 16 channels, can a resolution-aware coupled PDE–ODE transfer *unchanged* to 32/64/128, retain accuracy and numerical stability, and outperform resolution-neutral controls?

**Hypotheses (to be adjudicated by data, not asserted):**

- **H4a (discretization):** physically scaled coupling transfers better than unscaled local convolution at larger grids.
- **H4b (architectural advantage):** the scaled CCNN beats a competitive resolution-neutral Fourier/Neural ODE/attention baseline at 128 channels.
- **H4c (scope):** CCNN's advantage survives a nonlocal-dynamics negative control. A failure contradicts *generality*, not the correctness of diffusion models.
- **H4d (stability):** an unchanged CCNN operator remains numerically finite and its *relative spatial error* does not degrade catastrophically with resolution.
- **H4e (efficiency):** any error improvement justifies its additional evaluation cost. Because current solvers are unmatched, timing is exploratory; this hypothesis remains inconclusive until a solver-matched run.
- **H4f (repair transfer):** a repair trained at 16 transfers to finer grids; **not evaluated here**, reserved for RQ8 once RQ4 is understood.

### Claim boundary

These are **synthetic linear dynamics**, not a full nonlinear learned CCNN from the original paper and not a music or language benchmark. Strong structural matching to the local generator is deliberately tested **against** nonlocal controls and a flexible Fourier operator. One seed is only a smoke test; use the multi-seed configuration for substantive comparisons.


## 2. Proposed method and mathematical rationale

Let the periodic domain be \(s\in[0,2\pi)\). The local reference system is

\[
\partial_t u(s,t)=D\partial_{ss}u(s,t)-\lambda u(s,t)+\beta z(t),
\qquad \dot z(t)=\kappa\operatorname{mean}_s u(s,t)-\gamma z(t).
\]

For a uniform mesh, \(\Delta s=2\pi/N\), the **conservative nearest-neighbor diffusion operator** is

\[
\Delta_N u_i=\frac{u_{i-1}-2u_i+u_{i+1}}{(\Delta s)^2}.
\]

The \(1/(\Delta s)^2\) factor is essential. Reusing an *unscaled* three-tap stencil at a different grid size changes the modeled physical diffusion. We examine both versions and a **spectral derivative** with Fourier eigenvalue \(-k^2\), which is resolution-consistent on band-limited periodic functions.

Rather than using an explicit solver whose stable step size decreases quadratically with grid spacing, the linear experiment evolves spatial Fourier modes through a differentiable **matrix-exponential / spectral semigroup**. The mean Fourier mode is coupled to the global ODE via a 2×2 matrix exponential. This is a mathematically controlled *linear subcase* of the PDE–ODE CCNN formulation, **not** a general nonlinear CCNN solver.

For the negative control we replace diffusion with a nonlocal symmetric shift interaction,

\[
\partial_tu=\frac{a}{2}\big[u(s+\delta)+u(s-\delta)-2u(s)\big]-\lambda u+\beta z,
\]

whose Fourier symbol is \(a(\cos(k\delta)-1)\), not \(-Dk^2\). All models receive the same time interval and observations. A Fourier-mode neural operator is included because resolution-neutral representations already exist and CCNN should not be credited for a property shared by competing operators.

**Expected but unverified outcomes:** scaled stencils should avoid the large grid-dependent bias of an unscaled stencil on the *local* system; a Fourier neural operator may match or beat CCNN on both generators; generic ODEs may be less accurate on spatially coupled dynamics. If these expectations fail, the notebook records that failure as evidence for revising the proposed mathematics.


## 3. Executable experimental protocol

**Step 1 — Reproducibility and configuration.** The quick run uses one seed to verify the experiment. Set `FAST_DEV = False` for at least three seeds; 10–20 independent seeds and wider parameter sweeps are needed for confirmatory work. The evaluation includes horizons within training range and one longer-than-trained horizon. This study does **not** require Hugging Face downloads or credentials: it tests the architectural hypothesis before introducing WSP complexity.


In [ ]:
%pip -q install numpy pandas matplotlib torch nbformat

In [ ]:
import copy, json, math, os, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST_DEV = True   # Set False for 3 seeds and more training iterations in Colab.
SEEDS = [17] if FAST_DEV else [17, 29, 43]
EPOCHS = 12 if FAST_DEV else 30
TRAIN_COUNT = 144 if FAST_DEV else 256
VALID_COUNT = 64 if FAST_DEV else 96
TEST_COUNT = 64 if FAST_DEV else 96
BATCH_SIZE = 48
TRAIN_N = 16
TEST_NS = [16, 32, 64, 128]
DOMAIN = 2.0 * math.pi
MAX_HARMONIC = 6
TEST_HORIZONS = [0.35, 0.75, 1.4]
GENERATORS = ['local', 'nonlocal']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = Path('rq4_results')
OUTPUT_DIR.mkdir(exist_ok=True)
torch.set_num_threads(min(4, os.cpu_count() or 1))
print('Device:', DEVICE, '| seeds:', SEEDS, '| epochs:', EPOCHS)


### Method architecture and evaluation boundary

The diagram distinguishes the **proposed grid-aware operator** from the reference generator and baseline models. The shared parameters are learned at \(N=16\); the same checkpoint is evaluated at every larger resolution. No test-resolution observations inform training or model selection.

In [ ]:
from matplotlib.patches import FancyBboxPatch
fig,ax=plt.subplots(figsize=(13,4.1))
ax.set_xlim(0,12);ax.set_ylim(0,4);ax.axis('off')
def method_box(x,y,w,h,label,proposed=False):
    box=FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.12',
        edgecolor='black',linewidth=2.8 if proposed else 1.,facecolor='none',
        hatch='//' if proposed else None)
    ax.add_patch(box);ax.text(x+w/2,y+h/2,label,ha='center',va='center',fontsize=10)
def connector(x,y,u,v):
    ax.annotate('',xy=(u,v),xytext=(x,y),arrowprops=dict(arrowstyle='->',lw=1.5))
method_box(.2,2.5,2.1,.8,'Continuous field\n(shared coefficients)')
method_box(3.,2.5,1.8,.8,'16-channel\ntrain only')
method_box(5.5,2.5,2.0,.8,'GRID-AWARE CCNN\nshared weights',True)
method_box(8.3,2.5,1.6,.8,'Test at\n16 / 32 / 64 / 128')
method_box(10.5,2.5,1.2,.8,'Evidence\nledger')
for x,u in [(2.3,3.),(4.8,5.5),(7.5,8.3),(9.9,10.5)]:connector(x,2.9,u,2.9)
method_box(3.,.55,4.4,.7,'Controls: unscaled stencil · spectral · ODE · FNO-lite · attention')
connector(5.2,1.25,5.2,2.48)
method_box(8.3,.55,3.4,.7,'Nonlocal negative control + timing + stability')
connector(10.,1.25,10.,2.48)
fig.tight_layout();fig.savefig(OUTPUT_DIR/'rq4_method_architecture.png',dpi=175);plt.show()


### Step 2 — Construct a resolution-independent ground truth

Each continuous initial field has a random mean and up to six Fourier harmonics. Its *same coefficients* generate the 16/32/64/128 discretizations. The reference trajectories are computed analytically in Fourier space, including exact coupling of the zero mode and global state. Therefore, changing the grid **does not** silently change the underlying equation or introduce a coarse-grid reference solver error. The Nyquist limit at 16 channels is eight modes, above the six generated modes.

Training, validation and test initial coefficients are independently sampled. The training resolution is **always 16**. A test sample at 32, 64 or 128 is the same underlying continuous function sampled more densely—not a new physical problem. This intentionally isolates resolution transfer.


In [ ]:
TRUE = {'decay': 0.27, 'beta': 0.22, 'gamma': 0.34, 'kappa': 0.18,
        'diffusion': 0.055, 'nonlocal_strength': 0.78, 'nonlocal_shift': 1.12}

def draw_latents(count, seed, fixed_t=None):
    rng = np.random.default_rng(seed)
    k = np.arange(1, MAX_HARMONIC + 1)
    scale = .66 / np.sqrt(k)
    a = rng.normal(0, scale, size=(count, MAX_HARMONIC))
    b = rng.normal(0, scale, size=(count, MAX_HARMONIC))
    mean = rng.normal(0, .25, size=(count, 1))
    z0 = rng.normal(0, .48, size=(count, 1))
    t = (rng.uniform(.12, .85, size=(count, 1)) if fixed_t is None
         else np.full((count, 1), fixed_t))
    return {key: torch.tensor(value, dtype=torch.float32)
            for key, value in {'a':a, 'b':b, 'mean':mean, 'z0':z0, 't':t}.items()}

def sample_field(latents, n):
    x = torch.arange(n, dtype=torch.float32) * DOMAIN / n
    k = torch.arange(1, MAX_HARMONIC + 1, dtype=torch.float32)
    return (latents['mean']
          + (latents['a'][:, :, None] * torch.cos(k[:, None] * x[None, :])).sum(dim=1)
          + (latents['b'][:, :, None] * torch.sin(k[:, None] * x[None, :])).sum(dim=1))

def exact_coupled_step(u0, z0, t, decay, beta, gamma, kappa, rates):
    # A physically shared zero mode and a general diagonal nonzero Fourier symbol.
    n=u0.shape[-1]
    t=t.reshape(-1)
    A=torch.stack([torch.stack([-decay, beta]), torch.stack([kappa, -gamma])])
    expA=torch.matrix_exp(t[:, None, None] * A[None])
    initial=torch.stack([u0.mean(dim=-1), z0.reshape(-1)],dim=-1)
    future=torch.bmm(expA,initial[:,:,None])[:,:,0]
    modes=torch.fft.rfft(u0,dim=-1)
    evolved=modes[:,1:] * torch.exp(rates[None,1:] * t[:,None])
    dc=torch.complex(future[:,0:1]*n, torch.zeros_like(future[:,0:1]))
    out=torch.fft.irfft(torch.cat([dc,evolved],dim=-1),n=n,dim=-1)
    return out

def truth_rates(n, generator, device):
    modes=torch.fft.rfftfreq(n,d=DOMAIN/n).to(device)*(2*math.pi)
    if generator=='local':
        return -TRUE['decay'] - TRUE['diffusion'] * modes**2
    return (-TRUE['decay'] + TRUE['nonlocal_strength']
            * (torch.cos(modes*TRUE['nonlocal_shift'])-1.0))

def make_set(latents, n, generator, device=DEVICE):
    u0=sample_field(latents,n).to(device)
    z=latents['z0'].to(device)
    t=latents['t'].to(device)
    r=truth_rates(n,generator,device)
    with torch.no_grad():
        y=exact_coupled_step(u0,z,t,
              *[torch.tensor(TRUE[k],device=device) for k in ('decay','beta','gamma','kappa')],r)
    return (u0,z,t,y)

# Invariance check: the same continuous target sampled at 16 and 128 agrees at coarse grid sites.
probe=draw_latents(8, 991, fixed_t=.6)
coarse=make_set(probe,16,'local',torch.device('cpu'))[-1]
fine=make_set(probe,128,'local',torch.device('cpu'))[-1][:,::8]
assert torch.allclose(coarse,fine,atol=2e-5,rtol=2e-5), 'Reference violates resolution consistency'
print('Continuous-reference consistency | max abs difference:',(coarse-fine).abs().max().item())


### Step 3 — Build resolution-aware and resolution-neutral predictors

All models predict the same future field and receive the same forecast time and initial global state. The **proposed test mechanism** is `ccnn_scaled`: it uses the conservative grid-aware second derivative within a coupled PDE–ODE. `ccnn_naive` is the exact discretization counterexample (omits the mesh-width factor); `ccnn_spectral` tests whether the remaining error is due to the finite-volume approximation. The pointwise ODE, compact Fourier neural operator and spatial-coordinate attention are alternative resolution-neutral approaches.

Parameter counts are **reported**, not assumed equal. In this first structural experiment the baselines have substantially different capacities; that limitation prevents an unconditional accuracy claim. The Fourier model is an **FNO-style one-layer operator**, not a fully tuned production FNO. The *spectral CCNN* is an important near-oracle control because the local generator shares exactly its linear spectral parameterization. If it beats the finite-volume model, **H4b must not count as a success**, even when the proposed model wins against neural baselines.


In [ ]:
class LinearCoupledCCNN(nn.Module):
    def __init__(self, stencil='scaled'):
        super().__init__()
        assert stencil in ('scaled','naive','spectral')
        self.stencil=stencil
        self.raw_d=nn.Parameter(torch.tensor(-3.6))
        self.raw_decay=nn.Parameter(torch.tensor(-1.5))
        self.beta=nn.Parameter(torch.tensor(.09))
        self.raw_gamma=nn.Parameter(torch.tensor(-1.5))
        self.kappa=nn.Parameter(torch.tensor(.08))
    def forward(self,u0,z0,t):
        n=u0.shape[-1]
        modes=torch.arange(n//2+1,device=u0.device,dtype=u0.dtype)
        if self.stencil=='naive':
            eigen=4.0*torch.sin(math.pi*modes/n)**2
        elif self.stencil=='scaled':
            eigen=4.0*torch.sin(math.pi*modes/n)**2/(DOMAIN/n)**2
        else:
            eigen=modes**2
        d=F.softplus(self.raw_d);decay=F.softplus(self.raw_decay)
        gamma=F.softplus(self.raw_gamma)
        return exact_coupled_step(u0,z0,t,decay,self.beta,gamma,self.kappa,-decay-d*eigen)

class PointwiseNeuralODE(nn.Module):
    # Resolution-neutral dense pointwise derivative, same parameters for all N.
    def __init__(self,width=16,steps=6):
        super().__init__();self.steps=steps
        self.local=nn.Sequential(nn.Linear(3,width),nn.Tanh(),nn.Linear(width,1))
        self.global_net=nn.Sequential(nn.Linear(2,12),nn.Tanh(),nn.Linear(12,1))
    def rhs(self,u,z):
        mean=u.mean(-1,keepdim=True)
        inputs=torch.stack((u,mean.expand_as(u),z.expand_as(u)),dim=-1)
        return self.local(inputs).squeeze(-1),self.global_net(torch.cat((mean,z),dim=-1))
    def forward(self,u0,z0,t):
        u,z=u0,z0;dt=t/self.steps
        for _ in range(self.steps):
            a,b=self.rhs(u,z)
            c,d=self.rhs(u+dt*a/2,z+dt*b/2)
            e,f=self.rhs(u+dt*c/2,z+dt*d/2)
            g,h=self.rhs(u+dt*e,z+dt*f)
            u=u+dt[:, :1]*(a+2*c+2*e+g)/6
            z=z+dt*(b+2*d+2*f+h)/6
        return u

class FNOLite(nn.Module):
    # One FNO-style Fourier mixing layer, with variable-length spatial tokens.
    def __init__(self,width=8,modes=7):
        super().__init__();self.modes=modes
        self.lift=nn.Conv1d(5,width,1)
        self.spectral_real=nn.Parameter(torch.randn(width,width,modes)*.035)
        self.spectral_imag=nn.Parameter(torch.randn(width,width,modes)*.035)
        self.pointwise=nn.Conv1d(width,width,1)
        self.out=nn.Sequential(nn.Conv1d(width,width,1),nn.GELU(),nn.Conv1d(width,1,1))
    def forward(self,u0,z0,t):
        n=u0.shape[-1];x=torch.arange(n,device=u0.device,dtype=u0.dtype)*DOMAIN/n
        inp=torch.stack((u0,z0.expand_as(u0),t.expand_as(u0),
                     torch.sin(x)[None].expand_as(u0),torch.cos(x)[None].expand_as(u0)),dim=1)
        h=F.gelu(self.lift(inp));ft=torch.fft.rfft(h,dim=-1)
        m=min(self.modes,ft.shape[-1]);weights=torch.complex(self.spectral_real[:,:,:m],self.spectral_imag[:,:,:m])
        transformed=torch.einsum('bcm,com->bom',ft[:,:,:m],weights)
        out_ft=torch.cat((transformed,torch.zeros(*transformed.shape[:2],ft.shape[-1]-m,
                               device=ft.device,dtype=ft.dtype)),dim=-1)
        h=F.gelu(torch.fft.irfft(out_ft,n=n,dim=-1)+self.pointwise(h))
        return u0 + t*self.out(h).squeeze(1)

class CoordAttention(nn.Module):
    # Variable token count, known spatial coordinates and elapsed time.
    def __init__(self,hidden=16):
        super().__init__()
        self.embed=nn.Linear(5,hidden)
        self.attn=nn.MultiheadAttention(hidden,num_heads=2,batch_first=True)
        self.out=nn.Sequential(nn.LayerNorm(hidden),nn.Linear(hidden,hidden),nn.GELU(),nn.Linear(hidden,1))
    def forward(self,u0,z0,t):
        n=u0.shape[-1];x=torch.arange(n,device=u0.device,dtype=u0.dtype)*DOMAIN/n
        inp=torch.stack((u0,z0.expand_as(u0),t.expand_as(u0),
               torch.sin(x)[None].expand_as(u0),torch.cos(x)[None].expand_as(u0)),dim=-1)
        h=self.embed(inp);a,_=self.attn(h,h,h,need_weights=False)
        return u0+t*self.out(h+a).squeeze(-1)

MODEL_FACTORY={
  'ccnn_scaled':lambda:LinearCoupledCCNN('scaled'),
  'ccnn_naive':lambda:LinearCoupledCCNN('naive'),
  'ccnn_spectral':lambda:LinearCoupledCCNN('spectral'),
  'pointwise_ode':PointwiseNeuralODE,
  'fno_lite':FNOLite,
  'coord_attention':CoordAttention,
}
for key,ctor in MODEL_FACTORY.items():
    model=ctor().to(DEVICE)
    test=make_set(draw_latents(3,987,fixed_t=.5),32,'local')
    with torch.no_grad():prediction=model(*test[:3])
    assert prediction.shape==test[-1].shape and torch.isfinite(prediction).all(),key
    print(f'{key:19} trainable parameters: {sum(p.numel() for p in model.parameters()):5d}')


### Step 4 — Fit only at 16 channels

Each model sees exactly the same 16-channel training samples, timestamp range, and validation samples. Models use Adam and validation-based checkpoint restoration; hyperparameters are deliberately modest to make Colab feasible, and equal epochs are **not equal compute**. This first pass is an architectural diagnostic. The multi-seed run and explicit wall-clock measurements expose optimization confounds before any claim of superiority.


In [ ]:
def fit_model(model,train,val,epochs=EPOCHS):
    u,z,t,y=train;uv,zv,tv,yv=val
    optimizer=torch.optim.Adam(model.parameters(), lr=.035 if isinstance(model,LinearCoupledCCNN) else .003)
    best=float('inf');checkpoint=None;start=time.perf_counter()
    for epoch in range(epochs):
        model.train()
        order=torch.randperm(len(u),device=u.device)
        for idx in order.split(BATCH_SIZE):
            optimizer.zero_grad()
            pred=model(u[idx],z[idx],t[idx]);loss=(pred-y[idx]).square().mean()
            if not torch.isfinite(loss):raise FloatingPointError('Training loss diverged')
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),2.0);optimizer.step()
        model.eval()
        with torch.no_grad():
            value=float((model(uv,zv,tv)-yv).square().mean())
        if value<best:
            best=value;checkpoint=copy.deepcopy(model.state_dict())
    model.load_state_dict(checkpoint);model.eval()
    return best,time.perf_counter()-start

def evaluate_model(model,data):
    u,z,t,y=data
    with torch.no_grad():
        _=model(u[:min(8,len(u))],z[:min(8,len(u))],t[:min(8,len(u))])  # warmup
        if DEVICE.type=='cuda':torch.cuda.synchronize()
        started=time.perf_counter()
        out=torch.cat([model(u[i:i+32],z[i:i+32],t[i:i+32])
                       for i in range(0,len(u),32)],dim=0)
        if DEVICE.type=='cuda':torch.cuda.synchronize()
        elapsed=time.perf_counter()-started
        finite=bool(torch.isfinite(out).all())
        if not finite:return {'status':'diverged','mse':float('nan'),
                               'relative_l2':float('nan'),'eval_ms_per_sample':elapsed*1e3/len(u)}
        error=((out-y)**2).mean()
        # Uniform-grid normalized spatial L2 = sqrt(dx Σ error² / dx Σ truth²).
        rel=(torch.sum((out-y)**2,dim=-1)/(torch.sum(y*y,dim=-1)+1e-9)).sqrt().mean()
        return {'status':'evaluated','mse':float(error),
                'relative_l2':float(rel), 'eval_ms_per_sample':elapsed*1e3/len(u)}

RAW=[];FITS=[]
for gen in GENERATORS:
    for seed in SEEDS:
        torch.manual_seed(seed);np.random.seed(seed);random.seed(seed)
        train=make_set(draw_latents(TRAIN_COUNT,seed+1000),TRAIN_N,gen)
        val=make_set(draw_latents(VALID_COUNT,seed+2000),TRAIN_N,gen)
        test_latents={h:draw_latents(TEST_COUNT,seed+3000+j*111,fixed_t=h)
                     for j,h in enumerate(TEST_HORIZONS)}
        for name,factory in MODEL_FACTORY.items():
            model=factory().to(DEVICE)
            try:
                vloss,train_s=fit_model(model,train,val)
                FITS.append(dict(generator=gen,seed=seed,model=name,val_mse=vloss,
                     train_s=train_s,parameters=sum(p.numel() for p in model.parameters())))
                for n in TEST_NS:
                    for horizon in TEST_HORIZONS:
                        data=make_set(test_latents[horizon],n,gen)
                        metrics=evaluate_model(model,data)
                        RAW.append(dict(generator=gen,seed=seed,model=name,n=n,horizon=horizon,
                                        **metrics))
            except (RuntimeError,FloatingPointError,ValueError) as exc:
                FITS.append(dict(generator=gen,seed=seed,model=name,val_mse=np.nan,
                                 train_s=np.nan,parameters=sum(p.numel() for p in model.parameters()),
                                 failure=str(exc)))
                for n in TEST_NS:
                    for h in TEST_HORIZONS:
                        RAW.append(dict(generator=gen,seed=seed,model=name,n=n,horizon=h,
                              status='failed',mse=np.nan,relative_l2=np.nan,
                              eval_ms_per_sample=np.nan))
            print(f'{gen:8} seed={seed} model={name:19} train_status={FITS[-1].get("failure","ok")[:40]}')
            del model
            if DEVICE.type=='cuda':torch.cuda.empty_cache()
raw_df=pd.DataFrame(RAW);fit_df=pd.DataFrame(FITS)
print('\nValidation and compute:')
print(fit_df[['generator','seed','model','val_mse','train_s','parameters']].to_string(index=False))
raw_df.to_csv(OUTPUT_DIR/'rq4_raw_metrics.csv',index=False)
fit_df.to_csv(OUTPUT_DIR/'rq4_training_metrics.csv',index=False)


## 4. Evaluation: what improved, what did not, and why

**Step 5 — Compare resolution transfer and negative controls.** Figures use absolute error *and* differences against matched controls; a visually emphasized proposed method is not declared the winner. The long-horizon evaluation also tests whether temporal extrapolation and spatial transfer interact. The nonlocal generator is a planned failure test: a local diffusion operator should not be expected to fit every possible spatial interaction.


In [ ]:
valid_df=raw_df.query("status=='evaluated'").copy()
summary=(valid_df.groupby(['generator','model','n','horizon'],as_index=False)
         .agg(mse=('mse','mean'),sd=('mse','std'),relative_l2=('relative_l2','mean'),
              eval_ms=('eval_ms_per_sample','mean'),seeds=('seed','nunique')))
summary.to_csv(OUTPUT_DIR/'rq4_summary.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(13,5))
for ax,generator in zip(axes,GENERATORS):
    subset=summary[(summary.generator==generator)&(summary.horizon==.75)]
    for model in MODEL_FACTORY:
        d=subset[subset.model==model]
        if len(d):ax.plot(d.n,d.mse,marker='o',linewidth=2.6 if model=='ccnn_scaled' else 1.2,
                           linestyle='-' if model=='ccnn_scaled' else '--',label=model)
    ax.set_yscale('log');ax.set_xticks(TEST_NS)
    ax.set(title=f'{generator}: spatial transfer at t=0.75',xlabel='Spatial channels N',ylabel='Held-out field MSE ↓')
axes[0].legend(fontsize=8,ncol=2)
fig.tight_layout();fig.savefig(OUTPUT_DIR/'rq4_resolution_transfer.png',dpi=170);plt.show()

fig,ax=plt.subplots(figsize=(9,4.5))
for model in MODEL_FACTORY:
    d=summary[(summary.generator=='local')&(summary.n==128)&(summary.model==model)]
    if len(d):ax.plot(d.horizon,d.mse,marker='o',label=model,
                       linewidth=2.6 if model=='ccnn_scaled' else 1.2)
ax.set_yscale('log');ax.set(title='Local generator: temporal extrapolation at N=128',
                            xlabel='Forecast time (training ≤0.85)',ylabel='Held-out MSE ↓')
ax.legend(ncol=2,fontsize=8);fig.tight_layout()
fig.savefig(OUTPUT_DIR/'rq4_temporal_transfer.png',dpi=170);plt.show()

print('Mean long-horizon test MSE (t=1.4):')
print(summary[summary.horizon==1.4].pivot_table(index=['generator','model'],
              columns='n',values='mse').round(7).to_string())


### Step 6 — Explicit spatial-stability analysis

The stability limit of **forward Euler applied to a discrete heat equation** satisfies roughly \(\Delta t\le(\Delta s)^2/(2D)\). This is an analytic numerical fact, not an experimental claim of model performance. The spectral/matrix-exponential implementation used above avoids this explicit diffusion restriction in the linear case; it may nevertheless introduce FFT cost, and nonlinear extensions require additional solver analysis. Plotting the stability bound prevents a false impression that finer-grid transfer is computationally free.


In [ ]:
stable=[]
for n in TEST_NS:
    dx=DOMAIN/n
    dt_limit=dx*dx/(2*TRUE['diffusion'])
    stable.append({'n':n,'dx':dx,'explicit_euler_dt_max':dt_limit,
                   'minimum_steps_at_t_1_4':int(np.ceil(1.4/dt_limit))})
stability_df=pd.DataFrame(stable)
print(stability_df.to_string(index=False))
stability_df.to_csv(OUTPUT_DIR/'rq4_explicit_stability_bound.csv',index=False)


### Preliminary reference run: three-seed CPU result (15 November 2026)

The following is **measured evidence from a local CPU run** with `FAST_DEV=False`, seeds **17, 29, 43**, 30 epochs, 256 training initial fields per seed, and a 16-channel training grid. It is not a verified Colab run or a publication-grade confidence estimate. The live evaluation cells below regenerate results for the active configuration; use them as the authoritative outcome.

| Hypothesis | Three-seed toy observation | Classification |
|---|---|---|
| **H4a** — mesh-scaled vs unscaled stencil | At N=128, t=0.75: MSE **0.00255** vs **0.0848** | Supported *within this toy* |
| **H4b** — advantage over all resolution-neutral/spectral controls | Spectral coupled control reaches MSE **1.24×10⁻⁷**, better than scaled CCNN | Contradicted *within this toy* |
| **H4c** — generalizes to nonlocal dynamics | At N=128, t=0.75: spectral CCNN **0.0306**, FNO-lite **0.00278** | Contradicted *within this toy* |
| **H4d** — transfer without substantial relative error growth | Relative L2 rises from **0.0350** (N=16) to **0.0806** (N=128) for scaled CCNN | Contradicted by predeclared tolerance |
| **H4e** — efficient improvement | Execution strategies differ: exact semigroup vs RK4; timing not comparable as a solver efficiency test | Inconclusive |
| **H4f** — cross-resolution weight repair | No repair performed | Not tested |

**Interpretation.** This supports a **discretization-correctness benefit**, not a general architectural win. It also identifies a concrete mathematical weakness: parameters fitted to the finite-volume approximation at 16 channels do not necessarily remain optimal as the discrete eigenvalues approach their continuous counterparts at finer grids. The spectrally consistent operator removes this grid-induced error for the local generator, but its near-exact match is **built into the generator family**, so it is not proof of broad CCNN superiority.

**Decision for the next test:** Diagnose the coarse-grid dispersion mismatch before modifying the architecture. Compare a learned grid-consistent **spectral symbol** and an adaptive nonlocal flux against an independent neural operator, with matched parameter and optimization budgets. Do not add Hugging Face WSP until there is a reproducible defect and an identifiable parameter subset worth repairing.


## 5. Hypothesis evidence ledger (automatically generated)

This is the central scientific artifact of the notebook. A hypothesis may be **supported**, **contradicted**, **inconclusive**, or **not tested**, always qualified by experiment scope. One favourable dataset is not a mathematical proof; one toy counterexample can disprove a *universal* claim, but usually does not invalidate a narrower claim. The ledger uses preregistered thresholds and records the exact evidence it compares.

The quick Colab run provides **provisional** classifications only. Multi-seed means are necessary but not sufficient for statistical confidence. Future confirmatory work should report paired seed-level differences and uncertainty intervals, not only averages.


In [ ]:
def lookup(gen,model,n,h=.75,metric='mse'):
    d=summary[(summary.generator==gen)&(summary.model==model)&(summary.n==n)
              & (np.isclose(summary.horizon,h))]
    return float(d[metric].iloc[0]) if len(d)==1 else float('nan')

def valid(*vals):return all(np.isfinite(x) for x in vals)

def verdict(success, ok=True):
    if not ok:return 'INCONCLUSIVE'
    prefix='PROVISIONALLY ' if len(SEEDS)<3 else 'TOY-EVIDENCE '
    return prefix+('SUPPORTED' if success else 'CONTRADICTED')

ledger=[]
def add(code,hypothesis,evidence,status,next_action):
    ledger.append({'id':code,'hypothesis':hypothesis,'evidence':evidence,
                   'status':status,'next_action':next_action})

# H4a: difference from *not* scaling the same discrete stencil.
a=[(lookup('local','ccnn_scaled',n),lookup('local','ccnn_naive',n)) for n in (32,64,128)]
ok=all(valid(*x) for x in a)
support=ok and all(s<=.95*b for s,b in a)
add('H4a','Mesh-aware local CCNN beats the unscaled stencil at N>16',
    '; '.join(f'N{n}: scaled={s:.3g}, naive={b:.3g}' for n,(s,b) in zip((32,64,128),a)),
    verdict(support,ok), 'If supported, isolate finite-volume versus spectral discretization; otherwise diagnose operator learning and grid scaling.')

# H4b: structural benefit beyond a strong resolution-neutral baseline.
a=lookup('local','ccnn_scaled',128)
competitors={name:lookup('local',name,128) for name in ('ccnn_spectral','fno_lite','pointwise_ode','coord_attention')}
base_name=min(competitors,key=lambda name:competitors[name] if np.isfinite(competitors[name]) else float('inf'))
b=competitors[base_name];ok=valid(a,b)
add('H4b','Finite-volume CCNN improves on the best resolution-neutral or spectral control at N=128',
    f'CCNN={a:.4g}; best={base_name}:{b:.4g}; 5% relative margin required',
    verdict(a<=.95*b,ok),'If contradicted, do not claim spatial advantage; improve the operator only after isolating what the control models better.')

# H4c: explicit test of claimed universality using a nonlocal generator.
a=lookup('nonlocal','ccnn_spectral',128)
controls={name:lookup('nonlocal',name,128) for name in ('fno_lite','coord_attention','pointwise_ode')}
base_name=min(controls,key=lambda x:controls[x] if np.isfinite(controls[x]) else float('inf'))
b=controls[base_name];ok=valid(a,b)
add('H4c','Diffusion-based CCNN advantage is universal across local and nonlocal dynamics',
    f'Nonlocal CCNN spectral={a:.4g}; best {base_name}={b:.4g}',
    verdict(a<=.95*b,ok),'If contradicted, narrow the claim; investigate learned nonlocal flux or adaptive operator selection, not arbitrary added depth.')

# H4d: same parameters remain numerically finite with limited relative error growth.
coarse=lookup('local','ccnn_scaled',16,metric='relative_l2')
fine=lookup('local','ccnn_scaled',128,metric='relative_l2')
ok=valid(coarse,fine)
add('H4d','Unchanged scaled CCNN is stable and avoids severe error growth with resolution',
    f'relative L2 at 16={coarse:.4g}, 128={fine:.4g}; limit 1.25× + 0.01',
    verdict(fine<=1.25*coarse+.01,ok),
    'If contradicted, check training-grid bias, discrete operator eigenvalues, and solver stiffness.')

# H4e: usefulness must account for runtime; avoid declaring an advantage without error improvement.
sel=summary[(summary.generator=='local')&(summary.n==128)&(summary.horizon==.75)]
cc_ms=lookup('local','ccnn_scaled',128,metric='eval_ms')
base_ms=lookup('local','pointwise_ode',128,metric='eval_ms')
ok=valid(cc_ms,base_ms)
add('H4e','CCNN gains justify additional compute compared with pointwise ODE',
    f'CCNN={cc_ms:.4g} ms/sample; pointwise ODE={base_ms:.4g} ms/sample; '
    f'CCNN/local MSE={lookup("local","ccnn_scaled",128):.4g}; pointwise/local MSE={lookup("local","pointwise_ode",128):.4g}',
    'INCONCLUSIVE',
    'No speed claim until the semigroup and RK4 are compared under matched solver accuracy and wall-clock budget.')

add('H4f','A single weight-space edit transfers unchanged from N=16 to finer grids',
    'No repair intervention is included in the RQ4 experiment.', 'NOT TESTED',
    'Only after RQ4 mechanism is established: inject one controlled operator defect and test numerical/Adam/HF repair at N=16, then untouched N=32/64/128.')

ledger_df=pd.DataFrame(ledger)
print(ledger_df[['id','hypothesis','status','evidence']].to_string(index=False))
ledger_df.to_csv(OUTPUT_DIR/'rq4_hypothesis_ledger.csv',index=False)
(OUTPUT_DIR/'rq4_hypothesis_ledger.json').write_text(json.dumps(ledger,indent=2))


### Paired cross-seed differences and uncertainty

The table above uses mean error. To expose seed variability, we additionally compute **paired MSE differences** on the same test initial fields for each seed. The percentile bootstrap intervals below are exploratory with just three seeds and **must not be treated as definitive confidence bounds**. A proper confirmatory study needs at least 10 independent seeds, longer optimization and tuning-budget controls.

In [ ]:
def paired_effect(generator, model_a, model_b, n=128, horizon=.75):
    d=raw_df[(raw_df.generator==generator)&(raw_df.n==n)&
             np.isclose(raw_df.horizon,horizon)&
             (raw_df.model.isin([model_a,model_b]))]
    wide=d.pivot(index='seed',columns='model',values='mse').dropna()
    if model_a not in wide.columns or model_b not in wide.columns or len(wide)==0:
        return {'comparison':f'{model_a} - {model_b}','seeds':0,'status':'inconclusive'}
    delta=(wide[model_a]-wide[model_b]).to_numpy()
    rng=np.random.default_rng(12345)
    boot=rng.choice(delta,size=(5000,len(delta)),replace=True).mean(axis=1)
    return {'comparison':f'{model_a} - {model_b}','seeds':len(delta),
            'paired_mean_mse_difference':float(delta.mean()),
            'bootstrap_2_5_pct':float(np.quantile(boot,.025)),
            'bootstrap_97_5_pct':float(np.quantile(boot,.975)),
            'status':'exploratory bootstrap; insufficient seeds for confirmation'}

paired=[paired_effect('local','ccnn_scaled','ccnn_naive'),
        paired_effect('local','ccnn_scaled','ccnn_spectral'),
        paired_effect('nonlocal','ccnn_spectral','fno_lite')]
pd.DataFrame(paired).to_csv(OUTPUT_DIR/'rq4_paired_uncertainty.csv',index=False)
print(pd.DataFrame(paired).to_string(index=False))


## 6. Discussion, evidence synthesis and next research decision

**Step 7 — Generate an evidence-based build-up.** The conclusion below is computed from the results, not written as if we already know the winner. It separates what is *supported* from what is *contradicted*, what remains ambiguous, and what to do next. It does not use the phrase “mathematically proved” for an empirical result on a finite toy dataset.


In [ ]:
def build_conclusion(ledger):
    lines=['# RQ4 evidence-based conclusion',
           '',f'Run scope: seeds={SEEDS}; generators={GENERATORS}; N_train=16; N_test={TEST_NS};',
           f'epochs={EPOCHS}; horizons={TEST_HORIZONS}.',
           '', '## Supported within this experiment']
    selected=[x for x in ledger if 'SUPPORTED' in x['status']]
    lines += [f'- **{x["id"]}:** {x["hypothesis"]}. {x["evidence"]}' for x in selected] or ['- None demonstrated.']
    lines += ['','## Contradicted within this experiment']
    selected=[x for x in ledger if 'CONTRADICTED' in x['status']]
    lines += [f'- **{x["id"]}:** {x["hypothesis"]}. {x["evidence"]}' for x in selected] or ['- None contradicted.']
    lines += ['','## Inconclusive / not tested']
    selected=[x for x in ledger if x['status'] in ('INCONCLUSIVE','NOT TESTED')]
    lines += [f'- **{x["id"]}:** {x["status"]}. {x["evidence"]}' for x in selected] or ['- None.']
    lines += ['', '## Next experiments required by the observed evidence']
    for x in ledger:
        if x['status']!='NOT TESTED':
            lines.append(f'- **After {x["id"]}:** {x["next_action"]}')
    lines += ['', '**Decision gate:** Do not proceed to CCNN-enhanced attention (RQ1/T2) or '
              'cross-resolution HF weight repair (RQ8/T3) unless RQ4 identifies a structural '
              'advantage **beyond** resolution-neutral alternatives and acceptable compute cost.',
             '', '**Caution:** This is a finite synthetic study. The local generator shares '
              'the model’s linear physics. Results cannot establish general superiority, '
              'novelty or real-world transfer. A one-seed run is preliminary only.']
    return '\\n'.join(lines)

conclusion=build_conclusion(ledger)
print(conclusion.replace('\\n','\n'))
(OUTPUT_DIR/'RQ4_CONCLUSION_AND_NEXT_RESEARCH.md').write_text(conclusion.replace('\\n','\n'))
manifest={'research_question':'RQ4','charter':'RESEARCH_QUESTIONS_CCNN_ATTENTION_WEIGHT_REPAIR.md',
          'seeds':SEEDS,'epochs':EPOCHS,'train_n':TRAIN_N,'test_n':TEST_NS,
          'generators':GENERATORS,'train_count':TRAIN_COUNT,'test_count':TEST_COUNT,
          'horizons':TEST_HORIZONS,'device':str(DEVICE),'torch':torch.__version__,
          'scientific_status':'exploratory, not replicated'}
(OUTPUT_DIR/'rq4_run_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Saved metric CSVs, figures, hypothesis ledger, evidence-based conclusion and run manifest in',OUTPUT_DIR)


## 7. Conclusion and future research

This study is intended to determine **whether spatial operator consistency is a useful reason to augment attention with a coupled CCNN**. The empirical outcome is given in the generated evidence ledger and conclusion above; this manuscript does not predeclare it. The scaled finite-volume operator, spectral counterpart and naïve stencil distinguish discretization errors from representational advantages. The pointwise ODE, FNO-style and coordinate-attention controls test whether any advantage is unique. The nonlocal generator explicitly tests the limits of the inductive bias. A demonstrable gain solely against the unscaled stencil would be a **correct discretization lesson**, not yet a novel improvement over attention.

**Future research must follow evidence:**

1. If mesh scaling helps but FNO/attention still wins, investigate adaptive conservative fluxes, transport, or nonlocal kernels—and re-test against matched alternatives.
2. If the spectral model succeeds but finite-volume model fails, investigate discretization error and parameter estimation at the coarse training grid.
3. If spatial transfer works but temporal extrapolation fails, test observation-aware coupled dynamics and generic Neural ODEs with the same time information.
4. If no structural benefit is found, **do not** proceed to a large Transformer or Hugging Face WSP integration. Revisit the research premise.
5. Only after a reproducible RQ4 benefit: add the operator to an attention residual pathway (RQ1/T2) and test controlled weight repair across resolutions (RQ8/T3).

**References**

1. M. Habiba and B. A. Pearlmutter, *Continuous Convolutional Neural Networks: Coupled Neural PDE and ODE*, 2021. https://arxiv.org/abs/2111.00343
2. N. Kovachki et al., *Neural Operator: Learning Maps Between Function Spaces*, JMLR, 2023. https://arxiv.org/abs/2108.08481
3. Z. Li et al., *Fourier Neural Operator for Parametric Partial Differential Equations*, ICLR, 2021. https://arxiv.org/abs/2010.08895
4. R. T. Q. Chen et al., *Neural Ordinary Differential Equations*, NeurIPS, 2018. https://arxiv.org/abs/1806.07366
5. A. Gulati et al., *Conformer: Convolution-augmented Transformer for Speech Recognition*, 2020. https://arxiv.org/abs/2005.08100
6. R. Takehi et al., *Sparks of Weight-Space Programming*, 2026. https://rikiyat.github.io/blog/weight-space-programming/
7. [Research charter: CCNN × Attention × Verified Weight-Space Repair](https://github.com/mansura-habiba/ode_nn/blob/master/notebooks/RESEARCH_QUESTIONS_CCNN_ATTENTION_WEIGHT_REPAIR.md)
